In [1]:
!pip install -q transformers peft datasets accelerate

In [2]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
import sys
import os
from google.colab import drive

# 1. Monta il Drive
drive.mount('/content/drive')

# 2. Definisci i percorsi principali
BASE_DRIVE = '/content/drive/MyDrive/DeepLearning'
SRC_DIR = os.path.join(BASE_DRIVE, 'src')

# 3. Aggiungi la cartella 'src' ai percorsi di sistema di Python
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
    print(f"Directory {SRC_DIR} aggiunta al path di sistema.")

# 4. Ora puoi importare normalmente come se fossero librerie installate
from dataset import CLEVRDataset
from models import MultimodalCoT

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

# Impostiamo il device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Setup completato. Device in uso: {device}")

Mounted at /content/drive
Directory /content/drive/MyDrive/DeepLearning/src aggiunta al path di sistema.


✅ Setup completato. Device in uso: cuda


In [4]:
from torchvision import transforms

# 1. Inizializziamo il Tokenizer del T5 (o del modello LLM base che stai usando)
MODEL_NAME = "t5-base" # Cambialo se usi t5-base o un altro modello
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. Definiamo la pipeline di trasformazione per le immagini
# (Usa quella esatta che richiede il tuo vision_encoder)
transform = transforms.Compose([
    transforms.Resize((384, 384)), # Adatta alla dimensione richiesta dal tuo encoder
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Tokenizer e Transform pronti.")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

✅ Tokenizer e Transform pronti.


In [5]:
# Percorsi assoluti verso il tuo ecosistema "blindato"
BASE_DRIVE = '/content/drive/MyDrive/DeepLearning'
TRAIN_INDEX = os.path.join(BASE_DRIVE, 'indexes/train_index.json')
TRAIN_IMG_DIR = os.path.join(BASE_DRIVE, 'data/processed/images/train')

# Inizializziamo il Dataset
train_dataset = CLEVRDataset(
    index_path=TRAIN_INDEX,
    img_dir=TRAIN_IMG_DIR,
    tokenizer=tokenizer,
    transform=transform,
    stage='stage2'
)

# --- CREIAMO IL CUSTOM COLLATOR ---
def custom_collate_fn(batch):
    import torch
    return {
        # Impila le immagini in un unico tensore [batch_size, channels, height, width]
        'pixel_values': torch.stack([item['pixel_values'] for item in batch]),

        # Lascia i testi come semplici liste di stringhe (saranno processati dal tokenizer nella Cella 5)
        'input_text': [item['input_text'] for item in batch],
        'target_text': [item['target_text'] for item in batch],

        # Lascia i dati grezzi come semplice lista di dizionari (ignorati da PyTorch)
        'raw_item': [item['raw_item'] for item in batch]
    }

# Creiamo il DataLoader inserendo la nostra funzione personalizzata
BATCH_SIZE = 8
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    collate_fn=custom_collate_fn # <--- Il fix è qui!
)

print(f"✅ DataLoader pronto. Campioni totali: {len(train_dataset)} | Batch totali: {len(train_loader)}")

✅ DataLoader pronto. Campioni totali: 15000 | Batch totali: 1875


In [6]:
# Percorsi assoluti verso il tuo ecosistema "blindato"
BASE_DRIVE = '/content/drive/MyDrive/DeepLearning'
VAL_INDEX = os.path.join(BASE_DRIVE, 'indexes/val_index.json')
VAL_IMG_DIR = os.path.join(BASE_DRIVE, 'data/processed/images/val')

# Inizializziamo il Dataset
val_dataset = CLEVRDataset(
    index_path=VAL_INDEX,
    img_dir=VAL_IMG_DIR,
    tokenizer=tokenizer,
    transform=transform,
    stage='stage2'
)

# Creiamo il DataLoader inserendo la nostra funzione personalizzata
BATCH_SIZE = 8
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    collate_fn=custom_collate_fn # <--- Il fix è qui!
)

print(f"✅ DataLoader pronto. Campioni totali: {len(val_dataset)} | Batch totali: {len(val_loader)}")

✅ DataLoader pronto. Campioni totali: 1000 | Batch totali: 125


In [7]:
import os
import torch
from peft import LoraConfig, get_peft_model

# 1. Inizializza l'architettura base
model = MultimodalCoT().to(device)

# 💥 IL FIX: Iniezione della memoria visiva pre-addestrata (Stage 1)
print("🔄 Recupero della memoria visiva e di Cross-Attention dallo Stage 1...")
# Carichiamo il dizionario unico salvato alla fine del primo addestramento
stage1_custom_weights = torch.load(os.path.join(BASE_DRIVE, 'lora_adapter_final', 'custom_modules.pth'))

# Ripristiniamo i pesi esatti del ponte visivo
model.projector.load_state_dict(stage1_custom_weights['projector'])
model.cross_attention.load_state_dict(stage1_custom_weights['cross_attention'])
# Nota: layer_norm non serve ricaricarla se non la usi per condizionamenti specifici,
# ma se l'avevi salvata nel dizionario puoi aggiungere anche quella per sicurezza:
if 'layer_norm' in stage1_custom_weights:
    model.layer_norm.load_state_dict(stage1_custom_weights['layer_norm'])

print("✅ Conoscenza spaziale ripristinata con successo!")

# 2. Configurazione LoRA per il modello linguistico interno
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

# 3. Innestiamo LoRA nel sottomodello LLM
model.llm = get_peft_model(model.llm, lora_config)

# 💥 IL FIX FONDAMENTALE: Scongeliamo la vista per metterla a fuoco!
for param in model.projector.parameters():
    param.requires_grad = True
for param in model.cross_attention.parameters():
    param.requires_grad = True

# Stampiamo i parametri addestrabili per conferma
model.llm.print_trainable_parameters()

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.23GB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

🔄 Recupero della memoria visiva e di Cross-Attention dallo Stage 1...
✅ Conoscenza spaziale ripristinata con successo!
trainable params: 884,736 || all params: 223,788,288 || trainable%: 0.3953


In [8]:
from torch.optim import AdamW
import torch.amp
import os
import torch
from tqdm import tqdm

# ==========================================
# IPERPARAMETRI OTTIMIZZATI PER IL FINE-TUNING
# ==========================================
EPOCHS = 5
# 💥 FIX 1: Learning Rate ridotto di 10 volte per non distruggere i pesi pre-addestrati
LEARNING_RATE = 5e-5
ACCUMULATION_STEPS = 4
ADAPTER_DIR = os.path.join(BASE_DRIVE, 'lora_adapter_stage2')
os.makedirs(ADAPTER_DIR, exist_ok=True)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda')

best_val_loss = float('inf')

print("🚀 Inizio Addestramento e Validazione (Stage 2 - NaN Guard Attiva)...")

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    valid_batches = 0 # Contatore per la media reale
    optimizer.zero_grad()

    progress_bar = tqdm(train_loader, desc=f"Train Epoca {epoch+1}/{EPOCHS}")

    for step, batch in enumerate(progress_bar):
        pixel_values = batch['pixel_values'].to(device)

        inputs = tokenizer(batch['input_text'], return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        labels_encoding = tokenizer(batch['target_text'], return_tensors="pt", padding=True, truncation=True, max_length=5)

        labels = labels_encoding.input_ids
        labels[labels == tokenizer.pad_token_id] = -100
        labels = labels.to(device)

        with torch.amp.autocast('cuda'):
            outputs = model(
                pixel_values=pixel_values,
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                labels=labels
            )
            loss = outputs.loss / ACCUMULATION_STEPS

        # 💥 FIX 2: Lo Scudo Anti-NaN
        if torch.isnan(loss):
            print(f"\n⚠️ Rilevato spike a NaN al batch {step}. Salto l'aggiornamento per proteggere i pesi.")
            optimizer.zero_grad()
            continue # Saltiamo questo batch infetto

        scaler.scale(loss).backward()

        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_train_loss += (loss.item() * ACCUMULATION_STEPS)
        valid_batches += 1
        progress_bar.set_postfix({'loss': f"{(loss.item() * ACCUMULATION_STEPS):.4f}"})

    # Calcolo la media solo sui batch sani
    avg_train_loss = total_train_loss / valid_batches if valid_batches > 0 else float('inf')

    # ==========================================
    # 2. FASE DI VALIDATION (EARLY STOPPING)
    # ==========================================
    model.eval()
    total_val_loss = 0
    val_progress_bar = tqdm(val_loader, desc=f"Validazione Epoca {epoch+1}/{EPOCHS}")

    with torch.no_grad(): # Disattiviamo il calcolo dei gradienti per risparmiare memoria
        for batch in val_progress_bar:
            pixel_values = batch['pixel_values'].to(device)
            inputs = tokenizer(batch['input_text'], return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

            labels_encoding = tokenizer(batch['target_text'], return_tensors="pt", padding=True, truncation=True, max_length=16)
            labels = labels_encoding.input_ids
            labels[labels == tokenizer.pad_token_id] = -100
            labels = labels.to(device)

            # Inferenza veloce (nessun backward qui!)
            with torch.amp.autocast('cuda'):
                outputs = model(
                    pixel_values=pixel_values,
                    input_ids=inputs.input_ids,
                    attention_mask=inputs.attention_mask,
                    labels=labels
                )
                loss = outputs.loss

            total_val_loss += loss.item()
            val_progress_bar.set_postfix({'val_loss': f"{loss.item():.4f}"})

    avg_val_loss = total_val_loss / len(val_loader)

    print(f"\n🏁 Fine Epoca {epoch+1}")
    print(f"📉 Train Loss Media: {avg_train_loss:.4f} | 📊 Val Loss Media: {avg_val_loss:.4f}")

    # ==========================================
    # 3. SALVATAGGIO OTTIMIZZATO (Best Model)
    # ==========================================
    if avg_val_loss < best_val_loss:
        print(f"🌟 Miglioramento! Val Loss scesa da {best_val_loss:.4f} a {avg_val_loss:.4f}. Salvataggio pesi in corso...\n")
        best_val_loss = avg_val_loss

        # Salviamo la configurazione vincente
        model.llm.save_pretrained(ADAPTER_DIR)
        torch.save(model.projector.state_dict(), os.path.join(ADAPTER_DIR, 'projector.pth'))
        torch.save(model.cross_attention.state_dict(), os.path.join(ADAPTER_DIR, 'cross_attn.pth'))
    else:
        print(f"⚠️ Nessun miglioramento. I pesi non vengono sovrascritti (Best Val Loss rimane {best_val_loss:.4f}).\n")

    print("-" * 60)

🚀 Inizio Addestramento e Validazione (Stage 2 - NaN Guard Attiva)...


Train Epoca 1/5:   4%|▎         | 70/1875 [08:15<3:00:05,  5.99s/it, loss=1.9466]


⚠️ Rilevato spike a NaN al batch 69. Salto l'aggiornamento per proteggere i pesi.


Train Epoca 1/5:   4%|▍         | 77/1875 [09:00<3:14:30,  6.49s/it, loss=2.2934]


⚠️ Rilevato spike a NaN al batch 76. Salto l'aggiornamento per proteggere i pesi.


...


⚠️ Rilevato spike a NaN al batch 110. Salto l'aggiornamento per proteggere i pesi.


Train Epoca 5/5:  99%|█████████▉| 1857/1875 [11:24<00:06,  2.66it/s, loss=0.3643]


⚠️ Rilevato spike a NaN al batch 1856. Salto l'aggiornamento per proteggere i pesi.


Validazione Epoca 5/5: 100%|██████████| 125/125 [00:32<00:00,  3.87it/s, val_loss=0.5386]



🏁 Fine Epoca 5
📉 Train Loss Media: 0.4739 | 📊 Val Loss Media: 0.4659
🌟 Miglioramento! Val Loss scesa da 0.4737 a 0.4659. Salvataggio pesi in corso...

------------------------------------------------------------


In [9]:
# Definiamo la cartella di output finale
ADAPTER_DIR = os.path.join(BASE_DRIVE, 'lora_adapter_stage2')
os.makedirs(ADAPTER_DIR, exist_ok=True)

# 1. Salviamo il cervello linguistico (LoRA e Tokenizer)
model.llm.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# 2. SALVIAMO GLI OCCHI E IL RAGIONAMENTO (I moduli Custom)
custom_weights = {
    'projector': model.projector.state_dict(),
    'cross_attention': model.cross_attention.state_dict(),
    'layer_norm': model.layer_norm.state_dict()
}
torch.save(custom_weights, os.path.join(ADAPTER_DIR, 'custom_modules.pth'))

print(f"🎉 Modello e moduli visivi salvati con successo in: {ADAPTER_DIR}")

🎉 Modello e moduli visivi salvati con successo in: /content/drive/MyDrive/DeepLearning/lora_adapter_stage2
